In [0]:
from datetime import datetime
from pyspark.sql import Row

start_timestamp = datetime.now()

In [0]:

dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("catalog_name", "retail_lakehouse")
dbutils.widgets.text("source_pipeline_run_id", "")

In [0]:
environment = dbutils.widgets.get("environment").strip()
pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()
catalog_name = dbutils.widgets.get("catalog_name").strip()
source_pipeline_run_id = (
    dbutils.widgets.get("source_pipeline_run_id").strip()
)

In [0]:
import uuid

if not pipeline_run_id:
    pipeline_run_id = str(uuid.uuid4())

print(f"Environment            : {environment}")
print(f"Silver pipeline run ID : {pipeline_run_id}")
print(f"Bronze source run ID   : {source_pipeline_run_id or 'Latest unprocessed'}")
print(f"Catalog                : {catalog_name}")

In [0]:
bronze_table = (
    f"{catalog_name}.bronze.sales_transactions"
)

silver_table = (
    f"{catalog_name}.silver.sales_transactions"
)

rejected_table = (
    f"{catalog_name}.quarantine.sales_business_rejected"
)

audit_table = (
    f"{catalog_name}.audit.pipeline_runs"
)

silver_control_table = (
    f"{catalog_name}.audit.silver_processed_batches"
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_control_table}
(
    source_pipeline_run_id STRING,
    source_table STRING,
    target_table STRING,
    silver_pipeline_run_id STRING,
    records_read BIGINT,
    records_written BIGINT,
    records_rejected BIGINT,
    processed_timestamp TIMESTAMP,
    processing_status STRING
)
USING DELTA
COMMENT 'Tracks Bronze batches processed into Silver'
""")

In [0]:
from pyspark.sql import functions as F

display(
    spark.table(audit_table)
    .filter(
        (F.col("notebook_name") == "01_ingest_sales_bronze")
        & (F.col("run_status") == "SUCCESS")
    )
    .orderBy(F.col("created_timestamp").desc())
)

In [0]:
if not source_pipeline_run_id:
    raise ValueError(
        "source_pipeline_run_id is required. "
        "Provide the successful Bronze ingestion run ID."
    )

In [0]:
already_processed = (
    spark.table(silver_control_table)
    .filter(
        (F.col("source_pipeline_run_id") == source_pipeline_run_id)
        & (F.col("source_table") == bronze_table)
        & (F.col("processing_status") == "SUCCESS")
    )
    .limit(1)
    .count()
    > 0
)

if already_processed:
    dbutils.notebook.exit(
        f"Bronze batch already processed: {source_pipeline_run_id}"
    )

In [0]:
bronze_batch_df = (
    spark.table(bronze_table)
    .filter(
        F.col("_pipeline_run_id") == source_pipeline_run_id
    )
)

In [0]:
records_read = bronze_batch_df.count()

if records_read == 0:
    raise ValueError(
        f"No Bronze records found for run ID: "
        f"{source_pipeline_run_id}"
    )

print(f"Bronze records read: {records_read}")

In [0]:
normalized_df = (
    bronze_batch_df
    .withColumn(
        "transaction_id",
        F.upper(F.trim(F.col("transaction_id")))
    )
    .withColumn(
        "customer_id",
        F.upper(F.trim(F.col("customer_id")))
    )
    .withColumn(
        "product_id",
        F.upper(F.trim(F.col("product_id")))
    )
    .withColumn(
        "store_id",
        F.upper(F.trim(F.col("store_id")))
    )
    .withColumn(
        "status_code",
        F.upper(F.trim(F.col("status_code")))
    )
    .withColumn(
        "payment_method",
        F.initcap(F.trim(F.col("payment_method")))
    )
    .withColumn(
        "sales_channel",
        F.upper(F.trim(F.col("sales_channel")))
    )
    .withColumn(
        "promotion_id",
        F.upper(F.trim(F.col("promotion_id")))
    )
)

In [0]:
typed_df = (
    normalized_df
    .withColumn(
        "transaction_timestamp_parsed",
        F.try_to_timestamp(
            F.col("transaction_timestamp"),
            F.lit("yyyy-MM-dd HH:mm:ss")
        )
    )
    .withColumn(
        "quantity_parsed",
        F.col("quantity").try_cast("integer")
    )
    .withColumn(
        "unit_price_parsed",
        F.col("unit_price").try_cast("decimal(12,2)")
    )
    .withColumn(
        "discount_amount_parsed",
        F.coalesce(
            F.col("discount_amount").try_cast("decimal(12,2)"),
            F.lit(0).cast("decimal(12,2)")
        )
    )
)

In [0]:
validated_df = (
    typed_df
    .withColumn(
        "is_transaction_id_valid",
        F.col("transaction_id").isNotNull()
        & (F.length(F.col("transaction_id")) > 0)
    )
    .withColumn(
        "is_product_id_valid",
        F.col("product_id").isNotNull()
        & (F.length(F.col("product_id")) > 0)
    )
    .withColumn(
        "is_store_id_valid",
        F.col("store_id").isNotNull()
        & (F.length(F.col("store_id")) > 0)
    )
    .withColumn(
        "is_timestamp_valid",
        F.col("transaction_timestamp_parsed").isNotNull()
    )
    .withColumn(
        "is_quantity_valid",
        F.col("quantity_parsed").isNotNull()
        & (F.col("quantity_parsed") > 0)
    )
    .withColumn(
        "is_unit_price_valid",
        F.col("unit_price_parsed").isNotNull()
        & (F.col("unit_price_parsed") >= 0)
    )
    .withColumn(
        "is_discount_valid",
        F.col("discount_amount_parsed") >= 0
    )
    .withColumn(
        "is_status_valid",
        F.col("status_code").isin("S", "C", "R")
    )
)

In [0]:
# Creating Rejection Reason
validated_df = validated_df.withColumn(
    "rejection_reasons",
    F.array_compact(
        F.array(
            F.when(
                ~F.col("is_transaction_id_valid"),
                F.lit("INVALID_TRANSACTION_ID")
            ),
            F.when(
                ~F.col("is_product_id_valid"),
                F.lit("INVALID_PRODUCT_ID")
            ),
            F.when(
                ~F.col("is_store_id_valid"),
                F.lit("INVALID_STORE_ID")
            ),
            F.when(
                ~F.col("is_timestamp_valid"),
                F.lit("INVALID_TRANSACTION_TIMESTAMP")
            ),
            F.when(
                ~F.col("is_quantity_valid"),
                F.lit("INVALID_QUANTITY")
            ),
            F.when(
                ~F.col("is_unit_price_valid"),
                F.lit("INVALID_UNIT_PRICE")
            ),
            F.when(
                ~F.col("is_discount_valid"),
                F.lit("INVALID_DISCOUNT")
            ),
            F.when(
                ~F.col("is_status_valid"),
                F.lit("INVALID_STATUS_CODE")
            )
        )
    )
)

In [0]:
#Add the quality status:

validated_df = validated_df.withColumn(
    "data_quality_status",
    F.when(
        F.size(F.col("rejection_reasons")) == 0,
        F.lit("VALID")
    ).otherwise(F.lit("REJECTED"))
)

In [0]:
valid_df = validated_df.filter(
    F.col("data_quality_status") == "VALID"
)

rejected_df = validated_df.filter(
    F.col("data_quality_status") == "REJECTED"
)

In [0]:
records_rejected = rejected_df.count()
valid_before_dedup = valid_df.count()

print(f"Records read          : {records_read}")
print(f"Valid before dedup    : {valid_before_dedup}")
print(f"Rejected records      : {records_rejected}")

In [0]:
validated_df.select(
    F.sum(F.when(~F.col("is_transaction_id_valid"), 1).otherwise(0))
        .alias("invalid_transaction_id"),
    F.sum(F.when(~F.col("is_product_id_valid"), 1).otherwise(0))
        .alias("invalid_product_id"),
    F.sum(F.when(~F.col("is_store_id_valid"), 1).otherwise(0))
        .alias("invalid_store_id"),
    F.sum(F.when(~F.col("is_timestamp_valid"), 1).otherwise(0))
        .alias("invalid_timestamp"),
    F.sum(F.when(~F.col("is_quantity_valid"), 1).otherwise(0))
        .alias("invalid_quantity"),
    F.sum(F.when(~F.col("is_unit_price_valid"), 1).otherwise(0))
        .alias("invalid_unit_price"),
    F.sum(F.when(~F.col("is_discount_valid"), 1).otherwise(0))
        .alias("invalid_discount"),
    F.sum(F.when(~F.col("is_status_valid"), 1).otherwise(0))
        .alias("invalid_status")
).show()

In [0]:
display(
    validated_df.select(
        "transaction_id",
        "quantity",
        "transaction_timestamp",
        "status_code",
        "rejection_reasons",
        "data_quality_status"
    ).limit(30)
)

In [0]:
display(
    validated_df
    .filter(F.col("data_quality_status") == "REJECTED")
    .select(
        "transaction_id",
        "quantity",
        "transaction_timestamp",
        "status_code",
        "rejection_reasons",
        "data_quality_status"
    )
)

In [0]:
display(
    validated_df
    .filter(
        F.array_contains(
            F.col("rejection_reasons"),
            "INVALID_STATUS_CODE"
        )
    )
    .select(
        "transaction_id",
        "status_code",
        "rejection_reasons"
    )
)

In [0]:
#Write rejected business records

rejected_output_df = (
    rejected_df
    .withColumn(
        "_silver_pipeline_run_id",
        F.lit(pipeline_run_id)
    )
    .withColumn(
        "_source_bronze_run_id",
        F.lit(source_pipeline_run_id)
    )
    .withColumn(
        "_rejected_timestamp",
        F.current_timestamp()
    )
)

In [0]:
if records_rejected > 0:
    (
        rejected_output_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(rejected_table)
    )

    print(
        f"Rejected records written to {rejected_table}"
    )

In [0]:
#Deduplicate valid transactions

from pyspark.sql.window import Window

dedup_window = (
    Window
    .partitionBy("transaction_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc(),
        F.col("_source_file_modification_time").desc(),
        F.col("_record_hash").desc()
    )
)

In [0]:
deduplicated_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(dedup_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
deduplicated_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(dedup_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
valid_after_dedup = deduplicated_df.count()
duplicates_removed = valid_before_dedup - valid_after_dedup

print(f"Valid after dedup  : {valid_after_dedup}")
print(f"Duplicates removed : {duplicates_removed}")

In [0]:
#Map business values

business_df = (
    deduplicated_df
    .withColumn(
        "transaction_status",
        F.when(F.col("status_code") == "S", "SUCCESS")
        .when(F.col("status_code") == "C", "CANCELLED")
        .when(F.col("status_code") == "R", "RETURNED")
    )
)

In [0]:
#Normalizing Payment Method

business_df = business_df.withColumn(
    "payment_method_standardized",
    F.when(
        F.lower(F.col("payment_method")).contains("credit"),
        "CREDIT_CARD"
    )
    .when(
        F.lower(F.col("payment_method")).contains("debit"),
        "DEBIT_CARD"
    )
    .when(
        F.upper(F.col("payment_method")) == "UPI",
        "UPI"
    )
    .when(
        F.upper(F.col("payment_method")) == "CASH",
        "CASH"
    )
    .when(
        F.upper(F.col("payment_method")) == "WALLET",
        "WALLET"
    )
    .otherwise("OTHER")
)

In [0]:
# Create Derived Businedd Column:

business_df = (
    business_df
    .withColumn(
        "gross_amount",
        F.col("quantity_parsed")
        * F.col("unit_price_parsed")
    )
    .withColumn(
        "net_amount",
        (
            F.col("quantity_parsed")
            * F.col("unit_price_parsed")
        ) - F.col("discount_amount_parsed")
    )
    .withColumn(
        "transaction_date",
        F.to_date(F.col("transaction_timestamp_parsed"))
    )
    .withColumn(
        "order_year",
        F.year(F.col("transaction_timestamp_parsed"))
    )
    .withColumn(
        "order_month",
        F.month(F.col("transaction_timestamp_parsed"))
    )
    .withColumn(
        "order_week",
        F.weekofyear(F.col("transaction_timestamp_parsed"))
    )
    .withColumn(
        "return_flag",
        F.when(
            F.col("status_code") == "R",
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "cancellation_flag",
        F.when(
            F.col("status_code") == "C",
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)

In [0]:
#Handle financial treatment: 
#This means:
#Successful sale → positive revenue
#Returned sale → negative revenue
#Cancelled sale → zero revenue

business_df = (
    business_df
    .withColumn(
        "recognized_sales_amount",
        F.when(
            F.col("status_code") == "S",
            F.col("net_amount")
        )
        .when(
            F.col("status_code") == "R",
            -F.col("net_amount")
        )
        .otherwise(
            F.lit(0).cast("decimal(14,2)")
        )
    )
)

In [0]:
#Select the final Silver schema
silver_output_df = (
    business_df
    .select(
        "transaction_id",
        "customer_id",
        "product_id",
        "store_id",
        F.col("transaction_timestamp_parsed")
            .alias("transaction_timestamp"),
        F.col("quantity_parsed").alias("quantity"),
        F.col("unit_price_parsed").alias("unit_price"),
        F.col("discount_amount_parsed")
            .alias("discount_amount"),
        "gross_amount",
        "net_amount",
        "recognized_sales_amount",
        "status_code",
        "transaction_status",
        F.col("payment_method_standardized")
            .alias("payment_method"),
        "sales_channel",
        "promotion_id",
        "transaction_date",
        "order_year",
        "order_month",
        "order_week",
        "return_flag",
        "cancellation_flag",
        "_source_file_path",
        "_source_file_name",
        "_source_load_date",
        "_record_hash",
        "_ingestion_timestamp",
        F.lit(source_pipeline_run_id)
            .alias("_source_bronze_run_id"),
        F.lit(pipeline_run_id)
            .alias("_silver_pipeline_run_id"),
        F.current_timestamp()
            .alias("_silver_updated_timestamp")
    )
)

In [0]:
silver_output_df.printSchema()
display(silver_output_df.limit(20))

In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists(silver_table):
    (
        silver_output_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

    print(f"Created Silver table: {silver_table}")

else:
    silver_delta = DeltaTable.forName(
        spark,
        silver_table
    )

    (
        silver_delta.alias("target")
        .merge(
            silver_output_df.alias("source"),
            """
            target.transaction_id = source.transaction_id
            """
        )
        .whenMatchedUpdateAll(
            condition="""
            source._silver_updated_timestamp
            >= target._silver_updated_timestamp
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Merged records into: {silver_table}")

In [0]:
records_written = silver_output_df.count()

In [0]:
#Adding silver audit record

from datetime import datetime

audit_schema = spark.table(audit_table).schema

end_timestamp = datetime.now()

silver_audit_data = [{
    "pipeline_run_id": pipeline_run_id,
    "pipeline_name": "retail_sales_silver_transformation",
    "notebook_name": "01_transform_sales_silver",
    "source_system": "bronze_sales",
    "source_file_name": None,
    "source_file_path": bronze_table,
    "layer": "silver",
    "load_type": "INCREMENTAL",
    "run_status": "SUCCESS",
    "records_read": int(records_read),
    "records_written": int(records_written),
    "records_rejected": int(records_rejected),
    "start_timestamp": start_timestamp,
    "end_timestamp": end_timestamp,
    "error_message": None,
    "created_timestamp": datetime.now()
}]

silver_audit_df = spark.createDataFrame(
    silver_audit_data,
    schema=audit_schema
)

(
    silver_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(audit_table)
)

print("Silver pipeline audit written successfully.")

In [0]:
batch_control_record = [
    Row(
        source_pipeline_run_id=source_pipeline_run_id,
        source_table=bronze_table,
        target_table=silver_table,
        silver_pipeline_run_id=pipeline_run_id,
        records_read=records_read,
        records_written=records_written,
        records_rejected=records_rejected,
        processed_timestamp=datetime.now(),
        processing_status="SUCCESS"
    )
]

(
    spark.createDataFrame(batch_control_record)
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(silver_control_table)
)

print(
    f"Bronze batch registered successfully: "
    f"{source_pipeline_run_id}"
)

In [0]:
print(
    "Silver row count:",
    spark.table(silver_table).count()
)

In [0]:
display(
    spark.table(silver_table)
    .select(
        "transaction_id",
        "transaction_status",
        "quantity",
        "gross_amount",
        "discount_amount",
        "net_amount",
        "recognized_sales_amount",
        "return_flag",
        "cancellation_flag"
    )
    .limit(30)
)

In [0]:
duplicate_check_df = (
    spark.table(silver_table)
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate transaction IDs:",
    duplicate_check_df.count()
)

In [0]:
if spark.catalog.tableExists(rejected_table):
    display(
        spark.table(rejected_table)
        .select(
            "transaction_id",
            "quantity",
            "transaction_timestamp",
            "status_code",
            "rejection_reasons"
        )
    )